# SALM Fine-tuning 12 · Two languages in one sentence: Mandarin–English code-switching (ASCEND)

> **Task** speech → transcript that mixes 汉字 and English · **Data** ASCEND (8.8 h, Hong Kong conversations)
> **Metric** MER ↓ (mixed error rate) · **Result** 18.14% → **10.97%** (−39.5%) · **Time** ~1.5 h

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: "我觉得这个 project 的 deadline 太紧了"

Bilingual speakers switch languages mid-sentence, and ASR systems struggle with it. ASCEND records spontaneous
Mandarin–English conversations. The metric, MER, counts **Chinese characters and English words** as tokens: a WER where
汉字 count one by one. The base model is decent with a good prompt (~18% MER) but makes telling mistakes. The goal is a
transcriber that follows the speaker across languages and keeps the disfluencies:

```
REF   uh from uh                         BASE  அப்புறம்.                 ← a short English clip, decoded as Tamil
REF   twenty eighteen                    BASE  看T18。                   ← a stray character, and a digit
REF   uh in in in engineering school     BASE  In engineering school     ← disfluencies smoothed away
```

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data

In [ ]:
d = M / "ascend"
sh(f"{PYTHON} {FT}/prepare_benchmarks.py ascend", log="prep_ascend.log")
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")
for s in ("train", "dev", "test"):
    rows = [json.loads(l) for l in open(d / f"{s}.json")]
    print(f"{s:5s} {len(rows):5d} utterances, {sum(r['duration'] for r in rows) / 3600:.2f} h, speakers {sorted({r['speaker'] for r in rows})[:5]}")

## 2. Baseline: tell it what to expect

In [ ]:
TASK = "--task mer --max-tokens 256"
PROMPTS = {
    "p1": "Transcribe the audio.",
    "p2": "Transcribe the audio verbatim. The speaker may mix Mandarin Chinese and English; write Chinese in simplified "
          "characters and English words in Latin letters.",
    "p3": "Provide a verbatim transcript of the audio.",
}
base = {}
for p, text in PROMPTS.items():
    for s in ("dev", "test"):
        base[p, s] = 100 * evaluate(CKPT, d / f"{s}.json", f"cs_base_{p}_{s}", TASK, f"--prompt {q(text)}")["score"]
    print(f"{p}  dev MER {base[p, 'dev']:5.2f}%   test MER {base[p, 'test']:5.2f}%")
BEST_P = min(PROMPTS, key=lambda p: base[p, "dev"])  # prompt chosen on dev
BASELINE = base[BEST_P, "test"]  # its test score: the baseline fine-tuning is compared with
ORACLE = min(base[p, "test"] for p in PROMPTS)  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline = {BASELINE:.2f}% MER")
pairs = " ".join(f"{d}/{s}.json {d}/{s}_ft.json" for s in ("train", "dev", "test"))
sh(f"{PYTHON} {FT}/set_context.py --template {q(PROMPTS['p2'])} {pairs}")

Telling the model "the speaker may mix Mandarin and English" is worth ~2 points by itself. p2 is the prompt chosen on
dev and also the test-oracle.

The test set is only 0.9 h, and a single clip can fall into a greedy repetition loop
(`you know, just like, uh, you know, …`) depending on how vLLM batches the requests. On a test set this small, one
loop is worth about a point, so the baseline check allows ±1.5 (expect about 17.6-18.6%).

## 3. Config: a gentler learning rate

With 8.8 h of data, use LR 5e-5 rather than the series' usual 1e-4: on a corpus this small, 1e-4 overfits sooner.
Even at 5e-5, `val_loss` bottoms out at the first validation (step 90) and creeps up afterwards, while dev MER keeps
improving, so select by decoding, not by `val_loss`.

In [ ]:
CFG = WORK / "configs" / "cs.yaml"
sh(f"{PYTHON} {FT}/make_ft_config.py --checkpoint {CKPT} --prompt manifest --out {CFG} --exp-dir {EXP / 'cs'} "
   f"--train-manifest {d}/train_ft.json --val-manifest {d}/dev_ft.json --val-name cs_dev "
   f"--train-encoder --train-proj --lr 5e-5 --warmup-steps 40 --max-steps 450 "
   f"--batch-tokens 16000 --limit-train-batches 45 --val-every-n-epochs 2 --limit-val-batches 50 --save-top-k -1")

## 4. Train (~30 min)

In [ ]:
sh(f"{PYTHON} {FT}/run_training.py --config {CFG} --preflight --smoke --keep-steps 270 450")

## 5. Select on dev

Expected dev MER:

| model | dev MER |
|---|---:|
| base, prompt chosen on dev (p2) | 18.42% |
| step 270 / step 450 | 13.40% / 13.53% |
| **average(270, 450)** | **13.47%** |

The three candidates are within 0.15 points of each other, well inside the noise of a 0.9 h dev set. The average is
the more robust choice and is the selected model.

In [ ]:
dev = {}
for step in (270, 450):
    out = EXPORT / f"cs_s{step}"
    sh(f"{PYTHON} {FT}/export_checkpoint.py --exp-dir {EXP / 'cs'} --step {step} --base-checkpoint {CKPT} "
       f"--out {out} --delete-ckpt", log=f"export_cs_{step}.log")
    dev[step] = 100 * evaluate(out, d / "dev_ft.json", f"cs_s{step}_dev", TASK)["score"]
AVG = EXPORT / "cs_avg"
sh(f"{PYTHON} {FT}/average_checkpoints.py --src {EXPORT / 'cs_s270'} {EXPORT / 'cs_s450'} --dst {AVG} --reuse")
dev["avg"] = 100 * evaluate(AVG, d / "dev_ft.json", "cs_avg_dev", TASK)["score"]
for k, v in dev.items():
    print(f"{k:>4}: dev MER {v:.2f}%")

## 6. Test, once

In [ ]:
ft = evaluate(AVG, d / "test_ft.json", "cs_TEST", TASK)
guard = evaluate(AVG, M / "librispeech" / "test_clean.json", "cs_librispeech", "--task asr --normalizer english")
FT_MER = 100 * ft["score"]
print(f"test MER: base {BASELINE:.2f}% → fine-tuned {FT_MER:.2f}% ({(BASELINE - FT_MER) / BASELINE:+.1%} relative reduction); LibriSpeech {100 * guard['score']:.2f}%")
samples("cs_TEST", n=5)
result("12 baseline test MER", BASELINE, 18.14, 1.5)
result("12 fine-tuned test MER", FT_MER, 10.97, 1.0)

Expected results:

| | test MER | LibriSpeech WER |
|---|---:|---:|
| base, prompt chosen on dev (p2) | 18.14% | 1.54% |
| **fine-tuned (average of 2 checkpoints)** | **10.97%** | 1.50% |

**−39.5% relative.** Test is friendlier than dev (dev improves by −27%); with only two test speakers, this benchmark
swings. The fine-tuned model keeps the fillers (`uh`, `嗯`), writes numbers the way they were spoken, and doesn't
wander into Tamil.

## Tidy up

Keep the selected model; delete the other exports (~63 GB each) and any leftover training checkpoints.

In [ ]:
tidy(keep=AVG, name="cs")

## Takeaways

1. **Tell the model what to expect.** "The speaker may mix Mandarin and English" is worth 2 points.
2. **When `val_loss` bottoms out early on a small corpus, try a smaller LR** before anything fancier.
3. **Small test sets swing.** Two speakers gave −40% on test against −27% on dev. Report intervals.